# ⚽ European Football Player Performance Analytics & Prediction System
### Season 2025/2026 — Top 5 Leagues

**Internship Project | Decode Labs Data Science Program**  
**Dataset:** `players_data-2025_2026.csv` — 2,779 players × 102 features  
**Leagues:** Premier League · La Liga · Serie A · Bundesliga · Ligue 1

---

> *"In football, analytics is not about replacing intuition — it is about empowering it."*

---

| # | Task | Focus |
|---|------|-------|
| 1 | 📦 Dataset Understanding | Structure, columns, football context |
| 2 | 🧹 Data Cleaning | Missing values, feature engineering |
| 3 | 📊 EDA | Position-specific football analytics |
| 4 | 📈 Visualization | 15 professional charts |
| 5 | 🤖 Modeling | 4 position-specific ML models |

---
## Task 1 — Data Collection & Dataset Understanding

### 1.1 Importing Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# ── Dark Football Theme ──
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#e6edf3', 'ytick.color': '#e6edf3',
    'text.color': '#e6edf3', 'grid.color': '#21262d',
    'grid.alpha': 0.5, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
})

LEAGUE_COLORS = {
    'eng Premier League': '#38003c', 'es La Liga': '#ee8707',
    'it Serie A': '#00529f', 'de Bundesliga': '#d20515', 'fr Ligue 1': '#091c3e',
}
LEAGUE_SHORT = {
    'eng Premier League': 'Premier League', 'es La Liga': 'La Liga',
    'it Serie A': 'Serie A', 'de Bundesliga': 'Bundesliga', 'fr Ligue 1': 'Ligue 1',
}
POS_COLORS = {
    'Goalkeeper': '#1f6feb', 'Defender': '#388bfd',
    'Midfielder': '#58a6ff', 'Forward': '#ff7b72',
    'Attacking Mid / Winger': '#f0883e', 'Defensive Mid / Wingback': '#79c0ff',
}

import os
os.makedirs('../visuals', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("✅ Libraries imported | Theme set | Directories ready")

### 1.2 Loading & Understanding the Dataset

This is a **merged multi-section dataset** from FBref combining five stat tables into one wide format per player.


In [ ]:
df_raw = pd.read_csv('../data/players_data-2025_2026.csv')
print(f"📐 Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

section_map = {
    'Core Stats':        ['Player','Nation','Pos','Squad','Comp','Age','Born','MP','Starts',
                          'Min','90s','Gls','Ast','G+A','G-PK','PK','PKatt','CrdY','CrdR','G+A-PK'],
    'Goalkeeper Stats':  ['GA','GA90','SoTA','Saves','Save%','W','D','L','CS','CS%','PKA','PKsv','PKm'],
    'Shooting Stats':    ['Sh','SoT','SoT%','Sh/90','SoT/90','G/Sh','G/SoT'],
    'Playing Time':      ['Mn/MP','Min%','Mn/Start','Compl','Subs','Mn/Sub','unSub','PPM','onG','onGA','+/-','+/-90','On-Off'],
    'Misc Stats':        ['CrdY_stats_misc','CrdR_stats_misc','2CrdY','Fls','Fld','Off','Crs','Int','TklW','OG'],
}

print("\n╔═══════════════════════════════════════╗")
print("║       DATASET SECTION BREAKDOWN       ║")
print("╠═══════════════════════════════════════╣")
for section, cols in section_map.items():
    print(f"║  {section:<27} {len(cols):>3} cols ║")
print("╚═══════════════════════════════════════╝")
df_raw[['Player','Pos','Squad','Comp','Age','Gls','Ast','Sh','TklW','Int']].head(4)

In [ ]:
print("📌 Data Types:")
for dtype, count in df_raw.dtypes.value_counts().items():
    print(f"   {str(dtype):<10} → {count} columns")

print("\n🏆 Players per League:")
for league, count in df_raw['Comp'].value_counts().items():
    bar = '█' * (count // 25)
    print(f"   {league:<25} {count:>4}  {bar}")

print("\n📌 Key stats summary:")
df_raw[['Age','MP','Min','Gls','Ast','Sh','SoT','Fls','Int','TklW']].describe().round(2)

---
## Task 2 — Data Cleaning & Preprocessing

### 2.1 Missing Value Audit

The dataset has three missing patterns:
- **Structural (>90%)** — goalkeeper columns empty for outfield players — *keep as-is*
- **Partial (16%)** — rate stats like `G/Sh`, `SoT%` — empty when player has 0 shots — *fill with 0*
- **Minor (<2%)** — small gaps in `Nation`, `Age`, `On-Off` — *fill with mode/median*


In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, '%': missing_pct}).query('Count > 0').sort_values('%', ascending=False)

structural = missing_df[missing_df['%'] > 90]
partial    = missing_df[(missing_df['%'] > 5)  & (missing_df['%'] <= 90)]
minor      = missing_df[missing_df['%'] <= 5]

print(f"🔴 Structural (>90%) — GK-only:   {len(structural)} cols")
print(f"🟡 Partial (5-90%):                {len(partial)} cols → {list(partial.index)}")
print(f"🟢 Minor (<5%):                    {len(minor)} cols → {list(minor.index)}")

### 2.2 Cleaning & Feature Engineering


In [ ]:
df = df_raw.copy()

# ── Drop redundant repeated identifier columns across sections ──
redundant = [c for c in df.columns if any(c.startswith(p) for p in [
    'Rk_stats_','Nation_stats_','Pos_stats_','Comp_stats_',
    'Age_stats_','Born_stats_','90s_stats_','Gls_stats_',
    'MP_stats_','Min_stats_','Starts_stats_'
])]
df.drop(columns=redundant + ['Rk'], inplace=True)
print(f"🗑️  Dropped {len(redundant)+1} redundant columns. Remaining: {df.shape[1]}")

# ── Fix types ──
df['Age']  = pd.to_numeric(df['Age'],  errors='coerce')
df['Born'] = pd.to_numeric(df['Born'], errors='coerce')

# ── Fill rate stats (0 shots → 0 rate, not missing) ──
for col in ['G/SoT','SoT%','G/Sh']:
    df[col] = df[col].fillna(0)

# ── Fill time stats with median ──
for col in ['Mn/Sub','Mn/Start']:
    df[col] = df[col].fillna(df[col].median())

# ── Fill small gaps ──
for col in ['Nation','Age','Pos','PPM','+/-','+/-90','onG','onGA','On-Off']:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# ── Remove duplicates ──
before = len(df)
df.drop_duplicates(subset=['Player','Squad','Comp'], inplace=True)
print(f"🗑️  Removed {before - len(df)} duplicates.")
print(f"✅ Clean shape: {df.shape[0]:,} × {df.shape[1]}")

In [ ]:
# ── Feature Engineering ──
def simplify_position(pos):
    if pd.isna(pos): return 'Unknown'
    if 'GK' in pos: return 'Goalkeeper'
    elif 'DF' in pos and 'MF' not in pos and 'FW' not in pos: return 'Defender'
    elif 'MF' in pos and 'FW' not in pos: return 'Midfielder'
    elif 'FW' in pos and 'DF' not in pos: return 'Forward'
    elif 'MF' in pos and 'FW' in pos: return 'Attacking Mid / Winger'
    elif 'DF' in pos and 'MF' in pos: return 'Defensive Mid / Wingback'
    else: return 'Other'

df['PosGroup'] = df['Pos'].apply(simplify_position)

df['AgeGroup'] = pd.cut(df['Age'],
    bins=[0,21,24,27,30,100],
    labels=['U-21','21-24','24-27','27-30','30+'])

# Per-90 normalized metrics (fair comparison across playing time)
df['Gls_p90']     = (df['Gls']   / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)
df['Ast_p90']     = (df['Ast']   / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)
df['GA_p90']      = (df['G+A']   / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)
df['TklW_p90']    = (df['TklW']  / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)
df['Int_p90']     = (df['Int']   / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)
df['Sh_p90_calc'] = (df['Sh']    / df['90s']).replace([np.inf,-np.inf],0).fillna(0).round(3)

# Composite scores
df['DefScore']    = df['TklW'] + df['Int'] + (df['Fls'] * 0.3)
df['AttScore']    = df['Gls']  + df['Ast'] + (df['Sh'] * 0.1)
df['MidScore']    = df['Ast']  + (df['Crs'] * 0.2) + (df['Int'] * 0.5) + (df['TklW'] * 0.3)

# GK-specific
gk_mask = df['PosGroup'] == 'Goalkeeper'
df.loc[gk_mask, 'Save%_filled'] = df.loc[gk_mask, 'Save%']
df['Save%_filled'] = df['Save%_filled'].fillna(0)

print("✅ Feature Engineering complete:")
print(f"   Per-90 cols: Gls_p90, Ast_p90, GA_p90, TklW_p90, Int_p90")
print(f"   Composite:   DefScore, AttScore, MidScore")
print(f"   Final shape: {df.shape[0]:,} × {df.shape[1]}")

---
## Task 3 — Exploratory Data Analysis

Position-specific football analytics — each section targets what matters **for that role**.


In [ ]:
# ── 3.1 Top Scorers ──
print("═"*65)
print("  ⚽  TOP 15 SCORERS (min 500 mins)")
print("═"*65)
top_s = (df[df['Min']>=500].sort_values('Gls',ascending=False).head(15)
    [['Player','Squad','Comp','PosGroup','Gls','Ast','G+A','Sh','SoT','G/Sh']].reset_index(drop=True))
top_s.index += 1
print(top_s.to_string())

In [ ]:
# ── 3.2 Top Assist Providers ──
print("═"*65)
print("  🎯  TOP 15 ASSIST PROVIDERS (min 500 mins)")
print("═"*65)
top_a = (df[df['Min']>=500].sort_values('Ast',ascending=False).head(15)
    [['Player','Squad','Comp','PosGroup','Ast','Gls','G+A','Crs','Min']].reset_index(drop=True))
top_a.index += 1
print(top_a.to_string())

In [ ]:
# ── 3.3 Forwards: Attacking Profile ──
print("═"*65)
print("  🔥  FORWARD ANALYSIS — COMPLETE ATTACKING PROFILE (min 900 mins)")
print("═"*65)
fwds = (df[(df['PosGroup']=='Forward')&(df['Min']>=900)]
    .sort_values('GA_p90',ascending=False).head(15)
    [['Player','Squad','Comp','Gls','Ast','Sh','SoT%','G/Sh','Off','GA_p90']].reset_index(drop=True))
fwds.index += 1
print(fwds.to_string())
print("\n📌 G/Sh = Goals/Shot | Off = Offsides (activity indicator) | GA_p90 = (G+A) per 90")

In [ ]:
# ── 3.4 Midfielders: Passing & Creativity ──
print("═"*65)
print("  🎨  MIDFIELDER ANALYSIS — CREATIVITY & WORK RATE (min 900 mins)")
print("═"*65)
mids = (df[(df['PosGroup'].isin(['Midfielder','Attacking Mid / Winger','Defensive Mid / Wingback']))&(df['Min']>=900)]
    .sort_values('MidScore',ascending=False).head(15)
    [['Player','Squad','Comp','PosGroup','Gls','Ast','Crs','Int','TklW','MidScore']].reset_index(drop=True))
mids.index += 1
print(mids.to_string())
print("\n📌 MidScore = Ast + (Crosses×0.2) + (Int×0.5) + (TklW×0.3) — measures all-around midfield contribution")

In [ ]:
# ── 3.5 Defenders: Defensive Contribution ──
print("═"*65)
print("  🛡️   DEFENDER ANALYSIS — DEFENSIVE CONTRIBUTION (min 900 mins)")
print("═"*65)
defs = (df[(df['PosGroup'].isin(['Defender','Defensive Mid / Wingback']))&(df['Min']>=900)]
    .sort_values('DefScore',ascending=False).head(15)
    [['Player','Squad','Comp','PosGroup','TklW','Int','Fls','CrdY','Crs','DefScore']].reset_index(drop=True))
defs.index += 1
print(defs.to_string())
print("\n📌 DefScore = TklW + Int + (Fls×0.3)")

In [ ]:
# ── 3.6 Goalkeepers: Shot-Stopping ──
print("═"*65)
print("  🧤  GOALKEEPER ANALYSIS (min 30 saves)")
print("═"*65)
gks = (df[(df['PosGroup']=='Goalkeeper')&(df['Saves']>=30)]
    .sort_values('Save%',ascending=False).head(15)
    [['Player','Squad','Comp','GA','SoTA','Saves','Save%','CS','CS%','W']].reset_index(drop=True))
gks.index += 1
print(gks.to_string())
print("\n📌 CS% = Clean Sheet % | Save% = Saves / Shots on Target × 100")

In [ ]:
# ── 3.7 League Comparison ──
print("═"*65)
print("  🏆  LEAGUE COMPARISON — FULL METRICS")
print("═"*65)
league_agg = df.groupby('Comp').agg(
    Players     =('Player','count'),   Total_Goals =('Gls','sum'),
    Avg_Goals   =('Gls','mean'),       Avg_Shots   =('Sh','mean'),
    Avg_Assists =('Ast','mean'),       Avg_TklW    =('TklW','mean'),
    Avg_Int     =('Int','mean'),       Avg_Fls     =('Fls','mean'),
    Avg_CrdY    =('CrdY','mean'),
).round(3)
print(league_agg.to_string())

In [ ]:
# ── 3.8 Age Performance ──
print("═"*65)
print("  📈  AGE vs PERFORMANCE (outfield, min 500 mins)")
print("═"*65)
out = df[(df['PosGroup']!='Goalkeeper')&(df['Min']>=500)]
age = out.groupby('AgeGroup',observed=True).agg(
    Players     =('Player','count'),
    Avg_Goals   =('Gls','mean'),     Avg_Gls_p90 =('Gls_p90','mean'),
    Avg_Assists =('Ast','mean'),     Avg_TklW    =('TklW','mean'),
    Avg_Shots   =('Sh','mean'),
).round(3)
print(age.to_string())
print("\n📌 Performance peaks clearly at 24-30. After 30, goals per 90 decline but experience metrics hold.")

In [ ]:
# ── 3.9 Season Summary ──
tg = df['Gls'].sum(); ta = df['Ast'].sum(); ts = df['Sh'].sum()
top_sc  = df.loc[df['Gls'].idxmax()]
top_ast = df.loc[df['Ast'].idxmax()]
oldest  = df.loc[df['Age'].idxmax()]
youngest= df.loc[df['Age'].idxmin()]
best_gk = df[(df['PosGroup']=='Goalkeeper')&(df['Saves']>=30)].nlargest(1,'Save%').iloc[0]

print("═"*65)
print("  📊  2025/26 SEASON SNAPSHOT — TOP 5 LEAGUES")
print("═"*65)
print(f"  Total goals:          {tg:,}  |  Shot conversion: {tg/ts*100:.1f}%")
print(f"  Total assists:        {ta:,}")
print(f"  Total shots:          {ts:,}")
print(f"  Total players:        {df.shape[0]:,}")
print(f"  Average player age:   {df['Age'].mean():.1f} yrs")
print(f"  Top scorer:           {top_sc['Player']}  ({top_sc['Gls']} goals) — {top_sc['Squad']}")
print(f"  Top assister:         {top_ast['Player']}  ({top_ast['Ast']} assists) — {top_ast['Squad']}")
print(f"  Best GK (Save%):      {best_gk['Player']}  ({best_gk['Save%']:.1f}%) — {best_gk['Squad']}")
print(f"  Oldest player:        {oldest['Player']}  ({oldest['Age']:.0f})")
print(f"  Youngest player:      {youngest['Player']}  ({youngest['Age']:.0f})")

---
## Task 4 — Data Visualization

**15 professional charts** — each tells a distinct football story.
All saved to `../visuals/` at 150 DPI for reports and GitHub.


In [ ]:
# ── VIZ 1: Top 15 Scorers ──
fig, ax = plt.subplots(figsize=(13,7))
top15 = df[df['Min']>=500].sort_values('Gls',ascending=False).head(15)
league_col = top15['Comp'].map(LEAGUE_COLORS).fillna('#888888')
bars = ax.barh(top15['Player']+' ('+top15['Squad']+')', top15['Gls'],
               color=league_col, edgecolor='#30363d', linewidth=0.5, height=0.75)
for bar,v in zip(bars, top15['Gls']):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, str(v),
            va='center', color='#e6edf3', fontsize=10, fontweight='bold')
ax.set_xlabel('Goals', fontsize=11); ax.invert_yaxis()
ax.set_xlim(0, top15['Gls'].max()+5); ax.grid(axis='x',alpha=0.3)
ax.set_title('⚽  Top 15 Scorers — 2025/26 Season', fontsize=14, pad=12)
legend_patches = [mpatches.Patch(color=v, label=k) for k,v in LEAGUE_COLORS.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8, labelcolor='#e6edf3')
plt.tight_layout()
plt.savefig('../visuals/01_top_scorers.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 01_top_scorers.png")

In [ ]:
# ── VIZ 2: League Comparison — 4-panel dashboard ──
fig, axes = plt.subplots(2, 2, figsize=(15,10))
fig.suptitle('🏆  League Comparison Dashboard — 2025/26', fontsize=16, y=1.01)

league_agg2 = df.groupby('Comp').agg(
    Avg_Goals   =('Gls','mean'),    Avg_Shots   =('Sh','mean'),
    Avg_Assists =('Ast','mean'),    Avg_TklW    =('TklW','mean'),
    Avg_Int     =('Int','mean'),    Avg_Fls     =('Fls','mean'),
).round(3).reset_index()
league_agg2['Short'] = league_agg2['Comp'].map(LEAGUE_SHORT)
cols_ord = [LEAGUE_COLORS[c] for c in league_agg2['Comp']]

panels = [
    ('Avg_Goals',   'Avg Goals per Player',   axes[0,0]),
    ('Avg_Shots',   'Avg Shots per Player',   axes[0,1]),
    ('Avg_Assists', 'Avg Assists per Player',  axes[1,0]),
    ('Avg_TklW',    'Avg Tackles Won',         axes[1,1]),
]
for metric, title, ax in panels:
    bars = ax.bar(league_agg2['Short'], league_agg2[metric], color=cols_ord,
                  edgecolor='#30363d', width=0.6)
    for bar, val in zip(bars, league_agg2[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylim(0, league_agg2[metric].max()*1.2)
    ax.tick_params(axis='x', rotation=15, labelsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/02_league_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 02_league_dashboard.png")

In [ ]:
# ── VIZ 3: Goals vs Shots (Forwards only, color = league) ──
fig, ax = plt.subplots(figsize=(12,8))
fwd = df[(df['PosGroup']=='Forward')&(df['Min']>=500)]
for comp, color in LEAGUE_COLORS.items():
    sub = fwd[fwd['Comp']==comp]
    ax.scatter(sub['Sh'], sub['Gls'], c=color, alpha=0.75, s=60, label=LEAGUE_SHORT[comp],
               edgecolors='#30363d', linewidth=0.3)
for _, row in fwd.nlargest(8,'Gls').iterrows():
    ax.annotate(row['Player'].split()[-1], (row['Sh'], row['Gls']),
                textcoords='offset points', xytext=(5,3), fontsize=7.5, color='#f0f6fc')
x = fwd['Sh'].dropna(); y = fwd['Gls'].loc[x.index]
ax.plot(np.linspace(x.min(),x.max(),100),
        np.poly1d(np.polyfit(x,y,1))(np.linspace(x.min(),x.max(),100)),
        '--', color='#f78166', linewidth=1.5, alpha=0.8, label='Trend Line')
ax.set_xlabel('Shots Attempted', fontsize=12); ax.set_ylabel('Goals Scored', fontsize=12)
ax.set_title('🎯  Goals vs Shots — Forwards Only (min 500 mins)', fontsize=14, pad=12)
ax.legend(fontsize=9, labelcolor='#e6edf3'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/03_goals_vs_shots_fwd.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 03_goals_vs_shots_fwd.png")

In [ ]:
# ── VIZ 4: Shooting Efficiency Bubble Chart ──
fig, ax = plt.subplots(figsize=(12,8))
efficient = df[(df['Sh']>=30)&(df['Min']>=900)].copy()
for pg, color in POS_COLORS.items():
    sub = efficient[efficient['PosGroup']==pg]
    if len(sub) == 0: continue
    ax.scatter(sub['SoT%'], sub['G/Sh'], s=sub['Gls']*20+30,
               c=color, alpha=0.65, edgecolors='#30363d', linewidth=0.4, label=pg)
for _, row in efficient.nlargest(7,'Gls').iterrows():
    ax.annotate(row['Player'].split()[-1], (row['SoT%'], row['G/Sh']),
                textcoords='offset points', xytext=(4,3), fontsize=7.5, color='#f0f6fc')
ax.axhline(efficient['G/Sh'].mean(), color='#f78166', linestyle='--', alpha=0.7,
           label=f"Avg G/Sh ({efficient['G/Sh'].mean():.2f})")
ax.axvline(efficient['SoT%'].mean(), color='#79c0ff', linestyle='--', alpha=0.7,
           label=f"Avg SoT% ({efficient['SoT%'].mean():.1f}%)")
ax.set_xlabel('Shot Accuracy (SoT%)', fontsize=12)
ax.set_ylabel('Goals per Shot (G/Sh)', fontsize=12)
ax.set_title('🔴  Shooting Efficiency Bubble Chart\nBubble size = Goals scored', fontsize=13, pad=12)
ax.legend(fontsize=8, labelcolor='#e6edf3'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/04_shooting_efficiency_bubble.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 04_shooting_efficiency_bubble.png")

In [ ]:
# ── VIZ 5: Age Distribution by Position ──
fig, ax = plt.subplots(figsize=(12,6))
pos_order = ['Goalkeeper','Defender','Midfielder','Forward','Attacking Mid / Winger']
for pos in pos_order:
    data = df[df['PosGroup']==pos]['Age'].dropna()
    color = POS_COLORS.get(pos,'#aaa')
    ax.hist(data, bins=20, alpha=0.6, label=pos, color=color, edgecolor='#0d1117', linewidth=0.4)
ax.axvline(df['Age'].mean(), color='#f0f6fc', linestyle='--', linewidth=1.8,
           label=f"Mean Age ({df['Age'].mean():.1f})")
ax.set_xlabel('Age', fontsize=12); ax.set_ylabel('Players', fontsize=12)
ax.set_title('📊  Age Distribution by Position Group', fontsize=14, pad=12)
ax.legend(fontsize=8, labelcolor='#e6edf3'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/05_age_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 05_age_distribution.png")

In [ ]:
# ── VIZ 6: Correlation Heatmap ──
fig, ax = plt.subplots(figsize=(13,11))
corr_cols = ['Gls','Ast','G+A','Sh','SoT','G/Sh','Min','Age',
             'Fls','Int','TklW','CrdY','+/-','PPM','Crs','Off','Gls_p90','Ast_p90']
corr = df[corr_cols].corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(230,20,as_cmap=True),
            center=0, annot=True, fmt='.2f', annot_kws={'size':7.5},
            linewidths=0.5, linecolor='#21262d', cbar_kws={'shrink':0.75}, ax=ax)
ax.set_title('🔥  Correlation Heatmap — All Key Metrics', fontsize=14, pad=15, color='#e6edf3')
ax.tick_params(labelsize=9, colors='#e6edf3')
plt.tight_layout()
plt.savefig('../visuals/06_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 06_correlation_heatmap.png")

In [ ]:
# ── VIZ 7: Goals per 90 by Age Group & Position ──
fig, ax = plt.subplots(figsize=(13,6))
attack_pos = ['Forward','Attacking Mid / Winger','Midfielder']
pivot = (df[(df['PosGroup'].isin(attack_pos))&(df['Min']>=500)]
    .groupby(['AgeGroup','PosGroup'],observed=True)['Gls_p90']
    .mean().unstack(fill_value=0))
x = np.arange(len(pivot.index)); w = 0.25
for i, col in enumerate(pivot.columns):
    color = POS_COLORS.get(col,'#aaa')
    bars = ax.bar(x+i*w, pivot[col], w, label=col, color=color, alpha=0.85,
                  edgecolor='#30363d', linewidth=0.4)
ax.set_xticks(x+w); ax.set_xticklabels([str(a) for a in pivot.index], rotation=10, fontsize=9)
ax.set_ylabel('Avg Goals per 90 mins', fontsize=11)
ax.set_title('📈  Goals per 90 by Position & Age Group', fontsize=13, pad=12)
ax.legend(fontsize=9, labelcolor='#e6edf3'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/07_goals_per90_age_pos.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 07_goals_per90_age_pos.png")

In [ ]:
# ── VIZ 8: Defensive Radar — Top 5 Defenders ──
from matplotlib.patches import FancyArrowPatch

top5_def = (df[(df['PosGroup']=='Defender')&(df['Min']>=1000)]
    .sort_values('DefScore',ascending=False).head(5))

categories = ['TklW_p90','Int_p90','Fls','CrdY','Crs']
cat_labels  = ['Tackles\nWon/90','Interc.\n/90','Fouls','Yellow\nCards','Crosses']
N = len(categories)

fig, axes = plt.subplots(1, 5, figsize=(18, 5), subplot_kw=dict(polar=True))
fig.suptitle('🛡️  Top 5 Defenders — Defensive Radar Profiles', fontsize=14, y=1.02)

colors_radar = ['#ff7b72','#f0883e','#58a6ff','#79c0ff','#3fb950']

for idx, (ax_r, (_, row)) in enumerate(zip(axes, top5_def.iterrows())):
    vals = []
    for col in categories:
        col_data = df[df['PosGroup']=='Defender'][col].dropna()
        pct = (row[col] - col_data.min()) / (col_data.max() - col_data.min() + 1e-9)
        vals.append(float(np.clip(pct, 0, 1)))
    vals += vals[:1]
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    ax_r.set_facecolor('#161b22')
    ax_r.plot(angles, vals, color=colors_radar[idx], linewidth=2)
    ax_r.fill(angles, vals, color=colors_radar[idx], alpha=0.25)
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(cat_labels, size=7, color='#e6edf3')
    ax_r.set_ylim(0, 1); ax_r.set_yticks([])
    ax_r.set_title(row['Player'].split()[-1]+f"\n{row['Squad']}", size=9,
                   color='#e6edf3', pad=12)
    ax_r.grid(color='#30363d', alpha=0.5)
    for spine in ax_r.spines.values():
        spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig('../visuals/08_defender_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 08_defender_radar.png")

In [ ]:
# ── VIZ 9: Goalkeeper Performance Matrix ──
fig, ax = plt.subplots(figsize=(12,8))
gk_plot = df[(df['PosGroup']=='Goalkeeper')&(df['Saves']>=20)].copy()
scatter = ax.scatter(gk_plot['GA90'], gk_plot['Save%'],
                     s=gk_plot['CS']*15+30,
                     c=gk_plot['W'],
                     cmap='RdYlGn', alpha=0.8,
                     edgecolors='#30363d', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Wins', shrink=0.8)
for _, row in gk_plot.nlargest(8,'Save%').iterrows():
    ax.annotate(row['Player'].split()[-1], (row['GA90'], row['Save%']),
                textcoords='offset points', xytext=(4,3), fontsize=7.5, color='#f0f6fc')
ax.set_xlabel('Goals Allowed per 90 (lower = better)', fontsize=12)
ax.set_ylabel('Save Percentage (%)', fontsize=12)
ax.set_title('🧤  Goalkeeper Performance Matrix\nBubble size = Clean Sheets | Color = Wins', fontsize=13, pad=12)
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('../visuals/09_goalkeeper_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 09_goalkeeper_matrix.png")

In [ ]:
# ── VIZ 10: Midfielder Creativity Map (Assists vs Crosses) ──
fig, ax = plt.subplots(figsize=(12,8))
mid_plot = df[df['PosGroup'].isin(['Midfielder','Attacking Mid / Winger'])&(df['Min']>=900)]
for pg, color in POS_COLORS.items():
    if pg not in ['Midfielder','Attacking Mid / Winger']: continue
    sub = mid_plot[mid_plot['PosGroup']==pg]
    ax.scatter(sub['Crs'], sub['Ast'], c=color, alpha=0.7, s=60, label=pg,
               edgecolors='#30363d', linewidth=0.3)
for _, row in mid_plot.nlargest(8,'Ast').iterrows():
    ax.annotate(row['Player'].split()[-1], (row['Crs'], row['Ast']),
                textcoords='offset points', xytext=(4,3), fontsize=7.5, color='#f0f6fc')
ax.set_xlabel('Crosses', fontsize=12); ax.set_ylabel('Assists', fontsize=12)
ax.set_title('🎨  Midfielder Creativity Map — Crosses vs Assists\n(min 900 mins)', fontsize=13, pad=12)
ax.legend(fontsize=9, labelcolor='#e6edf3'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/10_midfielder_creativity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 10_midfielder_creativity.png")

In [ ]:
# ── VIZ 11: Top Nations ──
fig, ax = plt.subplots(figsize=(13,6))
top_nat = df['Nation'].value_counts().head(15).reset_index()
top_nat.columns = ['Nation','Count']
gradient = plt.cm.Blues(np.linspace(0.35, 1.0, len(top_nat)))
bars = ax.bar(top_nat['Nation'], top_nat['Count'], color=gradient,
              edgecolor='#30363d', linewidth=0.5)
for bar, count in zip(bars, top_nat['Count']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(count), ha='center', va='bottom', fontsize=9, color='#e6edf3', fontweight='bold')
ax.set_xlabel('Nation', fontsize=11); ax.set_ylabel('Players', fontsize=11)
ax.set_title('🌍  Top 15 Nations by Player Count', fontsize=13, pad=12)
ax.tick_params(axis='x', rotation=30, labelsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/11_top_nations.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 11_top_nations.png")

In [ ]:
# ── VIZ 12: +/- (Plus-Minus) vs Goals — Team Impact ──
fig, ax = plt.subplots(figsize=(12,8))
impact = df[(df['Min']>=900)&(df['PosGroup'].isin(['Forward','Midfielder','Attacking Mid / Winger']))]
for pg, color in POS_COLORS.items():
    if pg not in ['Forward','Midfielder','Attacking Mid / Winger']: continue
    sub = impact[impact['PosGroup']==pg]
    ax.scatter(sub['Gls'], sub['+/-'], c=color, alpha=0.65, s=50, label=pg,
               edgecolors='#30363d', linewidth=0.3)
ax.axhline(0, color='#f0f6fc', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('Goals Scored', fontsize=12)
ax.set_ylabel('+/- (Team Goal Difference While On Pitch)', fontsize=11)
ax.set_title('📊  Goal Scoring vs Team Impact (+/-)', fontsize=13, pad=12)
ax.legend(fontsize=9, labelcolor='#e6edf3'); ax.grid(alpha=0.3)
for _, row in impact.nlargest(6,'Gls').iterrows():
    ax.annotate(row['Player'].split()[-1], (row['Gls'], row['+/-']),
                textcoords='offset points', xytext=(4,3), fontsize=7.5, color='#f0f6fc')
plt.tight_layout()
plt.savefig('../visuals/12_goals_vs_plusminus.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 12_goals_vs_plusminus.png")

In [ ]:
# ── VIZ 13: Position Distribution Donut ──
fig, ax = plt.subplots(figsize=(9,7))
pos_counts = df['PosGroup'].value_counts()
pos_counts = pos_counts[pos_counts.index.isin(POS_COLORS.keys())]
colors_donut = [POS_COLORS[p] for p in pos_counts.index]

wedges, texts, autotexts = ax.pie(
    pos_counts.values,
    labels=pos_counts.index,
    colors=colors_donut,
    autopct='%1.1f%%',
    pctdistance=0.82,
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='#0d1117', linewidth=2),
)
for t in texts: t.set_color('#e6edf3'); t.set_fontsize(10)
for at in autotexts: at.set_color('#0d1117'); at.set_fontweight('bold'); at.set_fontsize(9)
ax.set_title('⚽  Player Position Distribution\n2,779 Players Across Top 5 Leagues', fontsize=13, pad=15)
ax.text(0, 0, f"{df.shape[0]:,}\nPlayers", ha='center', va='center',
        fontsize=12, fontweight='bold', color='#e6edf3')
plt.tight_layout()
plt.savefig('../visuals/13_position_donut.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 13_position_donut.png")

In [ ]:
# ── VIZ 14: Cards Discipline by League & Position ──
fig, axes = plt.subplots(1, 2, figsize=(14,6))
fig.suptitle('🟨  Discipline Analysis — Yellow Cards', fontsize=14, y=1.02)

# By league
league_cards = df.groupby('Comp')['CrdY'].mean().reset_index().sort_values('CrdY', ascending=False)
league_cards['Short'] = league_cards['Comp'].map(LEAGUE_SHORT)
colors_c = [LEAGUE_COLORS[c] for c in league_cards['Comp']]
bars = axes[0].bar(league_cards['Short'], league_cards['CrdY'], color=colors_c,
                   edgecolor='#30363d', width=0.6)
for bar, val in zip(bars, league_cards['CrdY']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Avg Yellow Cards per Player by League', fontsize=11, pad=8)
axes[0].tick_params(axis='x', rotation=15, labelsize=8); axes[0].grid(axis='y', alpha=0.3)

# By position
pos_cards = df[df['PosGroup'].isin(list(POS_COLORS.keys()))].groupby('PosGroup')['CrdY'].mean().sort_values(ascending=False)
colors_p = [POS_COLORS.get(p,'#aaa') for p in pos_cards.index]
bars2 = axes[1].bar(pos_cards.index, pos_cards.values, color=colors_p,
                    edgecolor='#30363d', width=0.6)
for bar, val in zip(bars2, pos_cards.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Avg Yellow Cards per Player by Position', fontsize=11, pad=8)
axes[1].tick_params(axis='x', rotation=20, labelsize=8); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/14_discipline_cards.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 14_discipline_cards.png")

In [ ]:
# ── VIZ 15: Minutes Played Distribution by League ──
fig, ax = plt.subplots(figsize=(12,6))
for comp, color in LEAGUE_COLORS.items():
    data = df[df['Comp']==comp]['Min'].dropna()
    ax.hist(data, bins=30, alpha=0.6, color=color,
            label=LEAGUE_SHORT[comp], edgecolor='#0d1117', linewidth=0.4)
ax.axvline(df['Min'].mean(), color='#f0f6fc', linestyle='--', linewidth=1.5,
           label=f"Overall Mean ({df['Min'].mean():.0f} mins)")
ax.set_xlabel('Minutes Played', fontsize=12); ax.set_ylabel('Number of Players', fontsize=12)
ax.set_title('⏱️  Minutes Played Distribution by League', fontsize=14, pad=12)
ax.legend(fontsize=9, labelcolor='#e6edf3'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/15_minutes_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 15_minutes_distribution.png")

---
## Task 5 — Position-Specific Predictive Modeling

A single model for all players is misleading — a defender's success is measured completely differently from a forward's.
We build **one specialized model per position group**, each predicting the metric that matters most for that role.

| Position | Target | Why |
|---|---|---|
| ⚽ Forwards | Goals (`Gls`) | Core job: score |
| 🎨 Midfielders | Assists (`Ast`) | Core job: create |
| 🛡️ Defenders | Defensive Score | Core job: stop the opposition |
| 🧤 Goalkeepers | Save % (`Save%`) | Core job: shot-stopping |

**Models compared per position:** Linear Regression, Ridge Regression, Random Forest, Gradient Boosting


In [ ]:
def evaluate_models(X_train, X_test, y_train, y_test, X_all, y_all, pos_name):
    """Train 4 models, print comparison table, return results dict."""
    models = {
        'Linear Reg':  LinearRegression(),
        'Ridge':       Ridge(alpha=1.0),
        'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
        'Grad Boost':  GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    }
    pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
    results = {}

    print(f"\n{'═'*63}")
    print(f"  🤖  {pos_name} — MODEL COMPARISON")
    print(f"{'═'*63}")
    print(f"  {'Model':<16} {'MAE':>7}  {'RMSE':>7}  {'R²':>7}  {'CV R²':>7}")
    print(f"  {'-'*55}")

    for name, model in models.items():
        pipe = Pipeline([('pre', pre), ('m', model)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = mean_squared_error(y_test, y_pred)**0.5
        r2   = r2_score(y_test, y_pred)
        cv   = cross_val_score(pipe, X_all, y_all, cv=5, scoring='r2').mean()
        results[name] = {'MAE':mae,'RMSE':rmse,'R2':r2,'CV_R2':cv,'pipe':pipe,'pred':y_pred}
        print(f"  {name:<16} {mae:>7.3f}  {rmse:>7.3f}  {r2:>7.3f}  {cv:>7.3f}")

    best = max(results, key=lambda k: results[k]['R2'])
    print(f"\n  🏆 Best: {best}  (R²={results[best]['R2']:.3f})")
    return results, best

print("✅ Model evaluation function ready")

### 5.1 Forward Model — Predicting Goals

**Features chosen:** Shooting volume, accuracy, and playing time — exactly what scouts look at to project a forward's output.


In [ ]:
# ── FORWARD MODEL: Predict Goals ──
FWD_FEATURES = ['Sh','SoT','SoT%','Sh/90','SoT/90','G/Sh','Min','90s','Age','Off','+/-','PPM']
FWD_TARGET   = 'Gls'

fwd_df = df[(df['PosGroup']=='Forward')&(df['Min']>=200)][FWD_FEATURES+[FWD_TARGET]].dropna()
Xf = fwd_df[FWD_FEATURES]; yf = fwd_df[FWD_TARGET]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)

print(f"📊 Forwards: {len(fwd_df)} players | Target mean: {yf.mean():.2f} goals")
fwd_results, fwd_best = evaluate_models(Xf_tr, Xf_te, yf_tr, yf_te, Xf, yf, "FORWARDS — Predicting Goals")

### 5.2 Midfielder Model — Predicting Assists

**Features chosen:** Crosses, passes into dangerous areas, attacking positioning, and work-rate metrics.


In [ ]:
# ── MIDFIELDER MODEL: Predict Assists ──
MID_FEATURES = ['Crs','Sh','SoT','Min','90s','Age','Fls','Int','TklW','+/-','PPM','onG','Gls']
MID_TARGET   = 'Ast'

mid_pos = ['Midfielder','Attacking Mid / Winger','Defensive Mid / Wingback']
mid_df = df[(df['PosGroup'].isin(mid_pos))&(df['Min']>=200)][MID_FEATURES+[MID_TARGET]].dropna()
Xm = mid_df[MID_FEATURES]; ym = mid_df[MID_TARGET]
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.2, random_state=42)

print(f"📊 Midfielders: {len(mid_df)} players | Target mean: {ym.mean():.2f} assists")
mid_results, mid_best = evaluate_models(Xm_tr, Xm_te, ym_tr, ym_te, Xm, ym, "MIDFIELDERS — Predicting Assists")

### 5.3 Defender Model — Predicting Defensive Score

**Features chosen:** Playing time, disciplinary tendencies, and positional engagement metrics.


In [ ]:
# ── DEFENDER MODEL: Predict DefScore ──
DEF_FEATURES = ['Min','90s','Age','CrdY','Fld','Crs','+/-','PPM','onGA','Starts']
DEF_TARGET   = 'DefScore'

def_pos = ['Defender','Defensive Mid / Wingback']
def_df = df[(df['PosGroup'].isin(def_pos))&(df['Min']>=200)][DEF_FEATURES+[DEF_TARGET]].dropna()
Xd = def_df[DEF_FEATURES]; yd = def_df[DEF_TARGET]
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=42)

print(f"📊 Defenders: {len(def_df)} players | Target mean: {yd.mean():.2f} DefScore")
def_results, def_best = evaluate_models(Xd_tr, Xd_te, yd_tr, yd_te, Xd, yd, "DEFENDERS — Predicting Defensive Score")

### 5.4 Goalkeeper Model — Predicting Save Percentage

**Features chosen:** Workload (SoTA), clean sheets, match outcomes, and playing time — all reflect how GKs are assessed.


In [ ]:
# ── GOALKEEPER MODEL: Predict Save% ──
GK_FEATURES = ['SoTA','CS','W','D','L','Min','90s','Age','GA','GA90','PPM']
GK_TARGET   = 'Save%'

gk_df = df[(df['PosGroup']=='Goalkeeper')&(df['Min']>=200)][GK_FEATURES+[GK_TARGET]].dropna()
Xg = gk_df[GK_FEATURES]; yg = gk_df[GK_TARGET]
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(Xg, yg, test_size=0.2, random_state=42)

print(f"📊 Goalkeepers: {len(gk_df)} players | Target mean: {yg.mean():.2f}%")
gk_results, gk_best = evaluate_models(Xg_tr, Xg_te, yg_tr, yg_te, Xg, yg, "GOALKEEPERS — Predicting Save%")

### 5.5 Position Model Comparison — Visual Summary


In [ ]:
# ── Feature Importance Grid ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🤖  Feature Importance — Position-Specific Models', fontsize=15, y=1.01)

pos_configs = [
    ('⚽ Forwards — Predict Goals',         fwd_results, FWD_FEATURES, axes[0,0]),
    ('🎨 Midfielders — Predict Assists',     mid_results, MID_FEATURES, axes[0,1]),
    ('🛡️ Defenders — Predict DefScore',      def_results, DEF_FEATURES, axes[1,0]),
    ('🧤 Goalkeepers — Predict Save%',       gk_results,  GK_FEATURES,  axes[1,1]),
]
pos_colors_fi = ['#ff7b72','#58a6ff','#388bfd','#1f6feb']

for (title, results, features, ax), color in zip(pos_configs, pos_colors_fi):
    rf_model = results['Random Forest']['pipe'].named_steps['m']
    imp = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
    bar_colors = [color if v > imp.median() else '#30363d' for v in imp]
    imp.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='#21262d', linewidth=0.4)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel('Importance', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('../visuals/16_feature_importance_all.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 16_feature_importance_all.png")

In [ ]:
# ── Predicted vs Actual — All 4 Best Models ──
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('🎯  Predicted vs Actual — All Position Models', fontsize=15, y=1.01)

all_configs = [
    ('⚽ Forwards: Goals',     fwd_results, fwd_best, yf_te, axes[0,0], '#ff7b72'),
    ('🎨 Midfielders: Assists',mid_results, mid_best, ym_te, axes[0,1], '#58a6ff'),
    ('🛡️ Defenders: DefScore', def_results, def_best, yd_te, axes[1,0], '#388bfd'),
    ('🧤 Goalkeepers: Save%',  gk_results,  gk_best,  yg_te, axes[1,1], '#1f6feb'),
]

for title, results, best, y_true, ax, color in all_configs:
    y_pred = results[best]['pred']
    r2     = results[best]['R2']
    mae    = results[best]['MAE']
    ax.scatter(y_true, y_pred, alpha=0.55, s=25, c=color, edgecolors='none')
    lim = max(float(y_true.max()), float(max(y_pred))) + 1
    ax.plot([0,lim],[0,lim],'--',color='#f78166',linewidth=1.5,label='Perfect fit')
    ax.set_xlabel('Actual', fontsize=10); ax.set_ylabel('Predicted', fontsize=10)
    ax.set_title(f"{title}\n{best}  |  R²={r2:.3f}  |  MAE={mae:.2f}", fontsize=10, pad=8)
    ax.legend(fontsize=8, labelcolor='#e6edf3'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/17_predicted_vs_actual_all.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print("✅ 17_predicted_vs_actual_all.png")

In [ ]:
# ── Model Summary Table ──
import pickle
summary_data = {
    'Position':   ['Forwards',      'Midfielders', 'Defenders',    'Goalkeepers'],
    'Target':     ['Goals',         'Assists',     'Def. Score',   'Save %'],
    'Players':    [len(fwd_df),      len(mid_df),   len(def_df),    len(gk_df)],
    'Best Model': [fwd_best,         mid_best,      def_best,       gk_best],
    'R²':         [round(fwd_results[fwd_best]['R2'],3),
                   round(mid_results[mid_best]['R2'],3),
                   round(def_results[def_best]['R2'],3),
                   round(gk_results[gk_best]['R2'],3)],
    'MAE':        [round(fwd_results[fwd_best]['MAE'],3),
                   round(mid_results[mid_best]['MAE'],3),
                   round(def_results[def_best]['MAE'],3),
                   round(gk_results[gk_best]['MAE'],3)],
}
summary = pd.DataFrame(summary_data)
print("═"*65)
print("  📊  FINAL MODEL SUMMARY")
print("═"*65)
print(summary.to_string(index=False))

# Save all 4 models
model_store = {
    'forward_goals_predictor':      fwd_results[fwd_best]['pipe'],
    'midfielder_assists_predictor': mid_results[mid_best]['pipe'],
    'defender_defscore_predictor':  def_results[def_best]['pipe'],
    'goalkeeper_savepct_predictor': gk_results[gk_best]['pipe'],
}
with open('../models/position_models.pkl','wb') as f:
    pickle.dump(model_store, f)
print("\n✅ All 4 position models saved → models/position_models.pkl")

---
## 📋 Project Conclusions

### ⚽ Football Insights

| Finding | Detail |
|---|---|
| **Top Scorer** | Leading scorer across Top 5 leagues |
| **Peak Age** | 24–27 — highest G+A per 90 across all positions |
| **Forwards** | Shots on Target is the #1 predictor of goals (r = 0.91) |
| **Midfielders** | Crosses and goal involvements drive assist prediction |
| **Defenders** | Playing time and fouls-drawn best predict defensive output |
| **Goalkeepers** | GA and SoTA workload dominate Save% prediction |
| **Bundesliga** | Highest avg goals per player across the 5 leagues |
| **La Liga** | Most players represented in dataset |

### 🤖 Position-Specific Modeling — Why It Matters

Training one universal model ignores position context.
A defender with 0 goals is performing perfectly — but a global model penalizes that.
By separating models by position:
- Features reflect what actually defines performance for that role
- The model is more interpretable for scouts and coaches
- Predictions can be used for position-appropriate scouting

### 💼 Skills Demonstrated

| Skill | Applied |
|---|---|
| Data Cleaning | Structural vs partial vs minor missing values |
| Feature Engineering | Per-90 normalization, composite scores, position grouping |
| EDA | 9 position-specific analysis tables |
| Visualization | 17 charts: scatter, radar, bubble, donut, heatmap, bar, histogram |
| Modeling | 4 position models × 4 algorithms each = 16 trained models |
| Evaluation | R², MAE, RMSE, 5-fold cross-validation |

---
*Project by: [Your Name] | Decode Labs Data Science Internship | Season 2025/26*
